# video-app lab 06 - FIRST REAL CLIP (guarded, High-RAM)
Confirmed runtime: Tesla T4 14.6 GB VRAM, 51.0 GB RAM, 193 GB disk. Models: AnimateDiff-Lightning
(4 steps) on SD1.5 base epiCRealism. Apache-2.0 / OpenRAIL-M. No key, no card, no cost.
Cell 1 is a hard guard: it stops before any download if the GPU is missing.


In [ ]:
import os, shutil, psutil, torch
vm = psutil.virtual_memory()
print(f'RAM   : {vm.used/1024**3:.1f} used / {vm.total/1024**3:.1f} GB total')
print('disk  :', round(shutil.disk_usage('/').free/1024**3,1), 'GB free | cuda:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise SystemExit('STOP: no GPU. Runtime > Change runtime type > T4 GPU + High-RAM > Save.')
p = torch.cuda.get_device_properties(0)
print('GPU   :', p.name, '| VRAM', round(p.total_memory/1024**3,1), 'GB  -> PREFLIGHT OK')


In [ ]:
import diffusers; print('diffusers', diffusers.__version__)
if tuple(int(x) for x in diffusers.__version__.split('.')[:2]) < (0,33):
    !pip -q install -U "diffusers>=0.33"
    import importlib, diffusers; importlib.reload(diffusers); print('upgraded to', diffusers.__version__)


In [ ]:
import gc, time, torch
def mem(tag):
    vm = psutil.virtual_memory()
    print(f'{tag:22s} RAM {vm.used/1024**3:5.1f}/51.0  VRAM {torch.cuda.memory_allocated()/1024**3:4.2f}/14.6 GB')
from diffusers import AnimateDiffPipeline, MotionAdapter, EulerDiscreteScheduler
DTYPE = torch.float16
mem('start')
adapter = MotionAdapter.from_pretrained('guoyww/animatediff-motion-adapter-v1-5-2', torch_dtype=DTYPE, low_cpu_mem_usage=True)
mem('adapter loaded')
pipe = AnimateDiffPipeline.from_pretrained('emilianJR/epiCRealism', motion_adapter=adapter,
        torch_dtype=DTYPE, low_cpu_mem_usage=True, use_safetensors=True)
mem('base loaded')
pipe.scheduler = EulerDiscreteScheduler.from_config(pipe.scheduler.config, timestep_spacing='trailing', beta_schedule='linear')
pipe.load_lora_weights('ByteDance/AnimateDiff-Lightning', weight_name='animatediff_lightning_4step_diffusers.safetensors', adapter_name='lightning')
pipe.set_adapters(['lightning'], adapter_weights=[1.0])
pipe.enable_vae_slicing(); pipe.enable_attention_slicing()
mem('lora applied')
pipe.to('cuda'); gc.collect()
mem('on cuda')


In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display
PROMPT = ('cinematic photo of a red fox trotting through fresh snow in a pine forest, '
          'morning light rays, shallow depth of field, steady camera, 35mm film')
t0 = time.time()
frames = pipe(prompt=PROMPT, negative_prompt='bad quality, blurry, distorted, watermark, text',
              num_frames=16, height=512, width=512, guidance_scale=1.0, num_inference_steps=4,
              generator=torch.Generator('cpu').manual_seed(42)).frames[0]
print('GENERATED in', round(time.time()-t0), 's')
mem('after generate')
export_to_video(frames, '/content/clip_06_fox.mp4', fps=8)
print('file MB:', round(os.path.getsize('/content/clip_06_fox.mp4')/1024**2,1))
display(Video('/content/clip_06_fox.mp4', embed=True, width=512))
from google.colab import files; files.download('/content/clip_06_fox.mp4')
